# SDK Analysis

Extract and summarize Genesys Cloud SDK API calls from **`TF_LOG=json`** output. Works with export, plan, or apply captures that contain `SDK DEBUG` hook lines.

Capture example (plan — use `export-tflog.log` or `apply-tflog.log` for other workflows):

```bash
export TF_LOG=json
export TF_LOG_PATH=plan-tflog.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
terraform plan
export TERRAFORM_LOG_PATH=plan-tflog.log
```

For response-time percentiles by endpoint, use **`log-chomper/`**. Run `_shared/whatisit.ipynb` if unsure.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path.cwd()
if (_nb_root.parent / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root.parent))
elif (_nb_root / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root))

import notebook_setup

notebook_setup.setup()

import pandas as pd
import commonlib.prep_sdk_data as prep_sdk_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg

In [ ]:
c = cfg.Config()
print(f"Reading terraform data from: {c.TERRAFORM_LOG_PATH}")
normalized_records = prep_sdk_data.load_normalized_records()
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No SDK DEBUG records found. Capture with TF_LOG=json during plan/apply/export. "
        "Confirm the file with _shared/whatisit.ipynb."
    )

print(sorted(df["debug_type"].drop_duplicates().tolist()))

df_sdk_request = df[df["debug_type"] == "SDK DEBUG REQUEST"].rename(
    columns={"timestamp": "request_timestamp"}
)
df_sdk_response = df[df["debug_type"] == "SDK DEBUG RESPONSE"].rename(
    columns={"timestamp": "response_timestamp"}
)

df_sdk_request_response = pd.merge(
    df_sdk_request[
        ["transaction_id", "invocation_method", "invocation_url", "sanitized_url", "request_timestamp"]
    ],
    df_sdk_response[
        ["transaction_id", "response_timestamp", "invocation_status_code", "invocation_retry_after"]
    ],
    on="transaction_id",
)

df_sdk_request_response["method_url"] = df_sdk_request_response.apply(
    lambda row: f"{row['invocation_method']} {row['sanitized_url']}", axis=1
)

## API Call Volume

In [ ]:
gencharts.generate_plt_by_method_url(
    df_sdk_request_response,
    df_sdk_request_response["method_url"],
    top_n=20,
)

## Top API Calls

In [ ]:
top_calls = df_sdk_request_response["method_url"].value_counts().head(20).reset_index()
top_calls.columns = ["method_url", "call_count"]
top_calls

## Sample Request/Response Pairs

In [ ]:
df_sdk_request_response.head(100)